<a href="https://colab.research.google.com/github/rubenchov/Python-Remotesensing/blob/main/TestAlphaEarth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
from google.colab import drive
import ee
import geemap
import pandas as pd
import numpy as np
import random
import cv2
from google.colab.patches import cv2_imshow
import os

Google Drive

In [ ]:
# Drive
drive.mount('/content/drive')
dirname = os.path.join(os.getcwd(), '/content/drive/MyDrive/Colab Notebooks/AlphaEarth/')

Mounted at /content/drive


Google Earth Engine (GEE)

In [ ]:
#IMPORTANT!
#1. USE COLAB WITH A GOOGLE PERSONAL ACCOUNT (NOT @elpoli.edu.co)
#2. SIGN IN TO https://developers.google.com/earth-engine
#3. CREATE A PROJECT, IN THIS CASE I NAMED IT "rubenchov". USE YOUR OWN.

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='rubenchov') #UPDATE ACCORDING TO THE NAME YOU CHOSE

#Function for Satellite Embedding

Define function

In [ ]:
def look_emb(LON1, LAT1, LON2, LAT2, date1, date2, bands_to_select):
  ROI = ee.Geometry.Rectangle(LON1, LAT1, LON2, LAT2)
  dataset = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL').filterDate(date1, date2).filterBounds(ROI)).first()
  # Return the unvisualized image with selected bands
  return dataset.select(bands_to_select).clip(ROI)

Define location, dates, download and export .tiff

In [ ]:
#Location
#lat1, long1, lat2, long2 = 5.35859, -72.41807, 5.31723, -72.37142 #Yopal
lat1, long1, lat2, long2 = 6.365982352088473, -75.66147753355916, 6.1065431974520115, -75.49922877302735 #Valle Aburrá

#Dates
year= '2020'
date1 = year + '-01-01'
date2 = year + '-12-31'

# Define bands to use for satellite embedding and clustering
bands_for_processing = ['A01', 'A16', 'A09']

#Download original image (unvisualized)
img = look_emb(long1, lat1, long2, lat2, date1, date2, bands_for_processing)

# Define visualization parameters for the original image
visualization = {'min': -0.3, 'max': 0.3, 'bands': bands_for_processing}
img_to_export_visualized = img.visualize(**visualization)

#Export original visualized image
export_path = os.path.join(dirname, year + '.tif')
geemap.ee_export_image(img_to_export_visualized, export_path, scale = 10, file_per_band=False)

Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/Colab Notebooks/AlphaEarth/2020.tif


K-means model

In [ ]:
# Define model
k = 5

# Define ROI for sampling and clustering (recreate here as it's not returned by look_emb)
ROI = ee.Geometry.Rectangle(long1, lat1, long2, lat2)

# Sample the image to create a FeatureCollection for training
# 'img' now holds the unvisualized image with selected bands from 'look_emb'
training_data = img.sample(region=ROI, scale=10, numPixels=1000, seed=0)

# Train the clusterer
model = ee.Clusterer.wekaKMeans(k).train(training_data)

# Predict clusters on the original image.
clustered = img.cluster(model)

# Export
export_path = os.path.join(dirname, year + '_clustered.tif')
geemap.ee_export_image(clustered.randomVisualizer(), export_path, scale=60, file_per_band=False)

Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/Colab Notebooks/AlphaEarth/2020_clustered.tif
